# Project 1 — Private PDF Q&A Bot with Ollama + LangChain

**Goal:** Build a small app that can answer questions about *your own PDF* — running **100% on your laptop**. No API key, no internet, no data leaving your machine.

**What you'll learn:**
1. How to talk to a local LLM using **Ollama** (via LangChain)
2. How to make **local embeddings** with `nomic-embed-text`
3. How to build a tiny **RAG pipeline** (retrieval + generation) that's fully offline
4. Why the "one-line swap" from `ChatOpenAI` → `ChatOllama` is such a big deal

**The mental model:**
> Ollama is *Docker for LLMs*. You `pull` a model, and it runs and serves it on `localhost:11434`. LangChain talks to it exactly the way it talked to OpenAI — same `.invoke()`, same `.stream()`, same chains.

---

## Step 0 — One-Time Setup (do this BEFORE running the notebook)

### 1. Install Ollama
- **Mac:** download from [ollama.com](https://ollama.com) (or `brew install ollama`)
- **Windows:** installer from [ollama.com](https://ollama.com)
- **Linux:** `curl -fsSL https://ollama.com/install.sh | sh`

### 2. Verify it's running
Open a terminal and run:
```bash
ollama --version
```

### 3. Pull the two models we'll use
```bash
ollama pull llama3.2             # Llama chat model
ollama pull nomic-embed-text     # embedding model — ~275 MB
```

> **Why `llama3.2:1b` and not `llama3.2`?** The 1B version is ~1.3 GB and works on 4–8 GB RAM laptops. If you have 16 GB+, feel free to use plain `llama3.2` (3B, better answers).

### 4. Make sure Ollama is running
On Mac/Windows the app runs in the background automatically. On Linux, run `ollama serve` in a separate terminal.

## Step 1 — Install the Python packages

In [ ]:
# Run once per environment
%pip install -q langchain langchain-ollama langchain-community pypdf faiss-cpu

## Step 2 — Hello, Local LLM 👋

Let's make sure we can talk to the model before we do anything fancy.

Under the hood, `ChatOllama` just makes an HTTP call to `http://localhost:11434`. That's it. That's the whole magic.

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0,   # deterministic — good for RAG & learning
)

response = llm.invoke("In one sentence: what is a large language model?")
print(response.content)

If you saw a sentence print above — **congratulations, you just ran an LLM on your own machine**. No API key. No internet needed after this point.

> 💡 Try turning off your Wi-Fi and re-running the cell. It still works.

## Step 3 — Load a PDF

We'll use LangChain's `PyPDFLoader`. For this demo, download any PDF and put it next to this notebook. Then update the filename below.

*Suggested demo PDFs:* your resume, a research paper, a product manual — anything you'd like to ask questions about.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# 👇 CHANGE THIS to your PDF filename
PDF_PATH = "Acceleration_Action_Information_Brief10th_July.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from {PDF_PATH}")
print("\n--- First 300 chars of page 1 ---")
print(pages[0].page_content[:300])

## Step 4 — Split the PDF into chunks

LLMs have a context window. We can't shove a 50-page PDF into one prompt. So we **chunk** the document into small overlapping pieces — later we'll retrieve only the chunks relevant to the question.

*Think of it like index cards:* one big book → many small cards → pull only the cards you need.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # ~500 chars per chunk
    chunk_overlap=50,    # a little overlap so we don't cut sentences awkwardly
)

chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")
print("\n--- Example chunk ---")
print(chunks[0].page_content)

## Step 5 — Turn chunks into vectors (embeddings) — locally

Every chunk becomes a vector (a list of numbers) that captures its meaning. Similar meanings → similar vectors.

We'll use **`nomic-embed-text`** served by Ollama — so the embeddings are also computed locally. **Zero cost, zero data leaving your machine.**

> 💰 If you did this with `OpenAIEmbeddings` on a big PDF, it'd cost real money. Here: $0.

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# FAISS = a fast in-memory vector database. Perfect for demos.
print("Building the vector store... (this may take a minute on the first run)")
vectorstore = FAISS.from_documents(chunks, embeddings)
print("✅ Vector store ready.")

## Step 6 — Ask a question (naive way first)

Before we build the full chain, let's just do the retrieval step by hand so you can *see* what's happening.

**Retrieval = "find the k most relevant chunks for this question."**

In [ ]:
question = "What is this document about?"   # 👈 change to something about YOUR pdf

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
relevant_chunks = retriever.invoke(question)

print(f"Found {len(relevant_chunks)} relevant chunks:\n")
for i, chunk in enumerate(relevant_chunks, 1):
    print(f"--- Chunk {i} ---")
    print(chunk.page_content[:200], "...\n")

## Step 7 — Build the full RAG chain

Now we wire it up: **question → retrieve chunks → stuff them into a prompt → local LLM answers.**

This is the same pattern you'd use with OpenAI. The **only** difference is `ChatOllama` and `OllamaEmbeddings` — everything else is identical.

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the user's question using ONLY the context below. "
               "If the answer isn't in the context, say you don't know.\n\nContext:\n{context}"),
    ("human", "{input}"),
])

doc_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, doc_chain)

print("✅ RAG chain ready. Ask away.")

In [ ]:
# 👇 Ask anything about your PDF
question = "Give me a 3-bullet summary of this document."

result = rag_chain.invoke({"input": question})
print("Q:", question)
print("\nA:", result["answer"])

### Try more questions

Change the `question` in the cell above and re-run. Some ideas:
- *"What are the main points on page 2?"*
- *"Explain [some concept from the pdf] in simple terms."*
- *"Is [some fact] mentioned in the document?"*

## Step 8 — Bonus: Streaming (the "ChatGPT effect")

Instead of waiting for the full answer, print tokens as they arrive. Same code you'd write for the OpenAI API.

In [ ]:
for chunk in llm.stream("Write a short poem about running AI on your own laptop."):
    print(chunk.content, end="", flush=True)

## 🎯 What you just built

You built a **fully local RAG system**:

```
Your PDF  →  chunks  →  local embeddings (nomic)  →  FAISS
                                                        ↓
              Your question  →  retrieve top-4 chunks  →  local LLM (llama3.2:1b)  →  answer
```

**Everything ran on your machine. Zero API cost. Works offline.**

### The big lesson (from the deck's Slide 25)

The code you wrote here is **the same code you'd write for OpenAI** — you just swapped:

| Before (paid, cloud) | After (free, local) |
|---|---|
| `from langchain_openai import ChatOpenAI` | `from langchain_ollama import ChatOllama` |
| `ChatOpenAI(model="gpt-4o-mini")` | `ChatOllama(model="llama3.2:1b")` |
| `from langchain_openai import OpenAIEmbeddings` | `from langchain_ollama import OllamaEmbeddings` |
| `OpenAIEmbeddings()` | `OllamaEmbeddings(model="nomic-embed-text")` |

That's it. **Two lines changed. Everything else works.** That's the whole point of the "layered architecture" — the Generator layer is swappable.

### Common issues
- **`Connection refused`** → Ollama isn't running. Start it (or run `ollama serve`).
- **Answers are slow** → normal on CPU. Try a smaller model or use a machine with a GPU.
- **RAG returns garbage** → make sure you built and queried the index with the *same* embedding model.

### Next
Head to Notebook 2 to see how **HuggingFace Transformers** compares — a different path with different tradeoffs.